# Speculative Decoding: a Small Model Drafts, a Large Model Verifies

> **Background**: autoregressive decoding generates tokens one by one. This serial dependency is a hard constraint. But can we "get multiple tokens per large-model forward"?
>
> Goal for this part: **understand speculative decoding: let a small draft model guess multiple tokens, then let the large target model verify them in one shot.**

One-sentence intuition:
**A cheap small model guesses 5 tokens; the expensive large model checks them all at once. Keep the correct prefix; when it fails, discard the rest and continue.**


In [ ]:
import torch
import torch.nn.functional as F

torch.manual_seed(42)

### 1. Why can't we generate multiple tokens at once?

Recall autoregressive generation:

```
Step 1: input [BOS]               -> predict token_1
Step 2: input [BOS, tok_1]        -> predict token_2
Step 3: input [BOS, tok_1, tok_2] -> predict token_3
```

Each step depends on the previous result. This is a **serial dependency**.

But wait: if we could **guess** token_1, we could compute token_2 earlier.

```
Normal:    compute tok_1 -> wait -> compute tok_2 -> wait -> compute tok_3
Speculate: guess tok_1=5, then compute tok_2 (assuming tok_1=5), and tok_3 (assuming tok_1=5,tok_2=3)
           If the guess is right, one forward can give multiple tokens.
```

That is the core intuition behind speculative decoding.


### 2. The full speculative decoding workflow

We need two models:
- **Draft model**: small, fast, cheap (e.g. 0.5B)
- **Target model**: large, slow, expensive (e.g. 70B)

Three steps:

```
+-------------------------------------------------------------+
| Step 1: Draft model proposes K tokens                        |
|                                                             |
| input:  [BOS, today, weather]                               |
|   -> draft autoregressively generates K tokens               |
| guess:  [is, great, today, !]   (K=4)                        |
|                                                             |
+-------------------------------------------------------------+
| Step 2: Target model verifies in one forward                 |
|                                                             |
| input:  [BOS, today, weather, is, great, today, !]           |
|   -> one target forward                                      |
| output: probabilities for each position                      |
|                                                             |
| check:                                                       |
|   pos 3: draft guessed "is"    -> target also likes "is"    [ok] |
|   pos 4: draft guessed "great" -> target also likes "great" [ok] |
|   pos 5: draft guessed "today" -> target prefers something else [x] |
|                                                             |
+-------------------------------------------------------------+
| Step 3: Accept / reject                                      |
|                                                             |
| accept the correct prefix guesses (e.g. first 2 tokens)      |
| reject the rest after the first failure                       |
| sample the next token from the target distribution           |
|                                                             |
| result: one target forward yields multiple tokens            |
+-------------------------------------------------------------+
```


### 3. Key question: how do we decide accept vs reject?

It is not simply "does the draft token equal the target argmax".

Instead we use a **probability ratio**.
For each draft-proposed token `x`:

```
p_target(x) = probability assigned by the target model
p_draft(x)  = probability assigned by the draft model

accept_prob = min(1, p_target(x) / p_draft(x))

if p_target(x) >= p_draft(x): always accept
else: accept with probability p_target/p_draft
```

Intuition:
- If the target is at least as confident as the draft, accept.
- If the draft is overconfident but the target disagrees, we may reject.

This rule guarantees: **the final output distribution matches pure target-model sampling** (provable).


In [ ]:
# Simulate the accept/reject logic in speculative decoding
def speculative_accept(draft_probs, target_probs, draft_tokens):
    """Decide which draft tokens are accepted.

    Args:
        draft_probs: Draft model probabilities for each proposed token [K]
        target_probs: Target model probabilities for each proposed token [K]
        draft_tokens: Draft-proposed token ids [K]

    Returns:
        A list of accepted token ids.
    """
    accepted = []
    for dp, tp, tok in zip(draft_probs, target_probs, draft_tokens):
        # Acceptance probability
        accept_prob = min(1.0, tp / dp) if dp > 0 else 0.0

        # Accept with probability accept_prob
        if torch.rand(1).item() < accept_prob:
            accepted.append(tok)
        else:
            # Reject: stop here and discard the remaining tokens
            break
    return accepted

# One simulated speculative decoding step
print("=== Simulated speculative decoding ===")
print()

# Draft proposes 4 tokens + its own confidence
# (In reality these would come from the draft model's next-token distribution.)
draft_tokens = [15, 23, 8, 42]
draft_probs = [0.8, 0.7, 0.6, 0.5]

# Target verifies with its probabilities on the same tokens
target_probs = [0.9, 0.8, 0.2, 0.1]

print(f"Draft proposed tokens: {draft_tokens}")
print(f"Draft probabilities:   {draft_probs}")
print(f"Target probabilities:  {target_probs}")
print()

for i in range(len(draft_tokens)):
    ratio = target_probs[i] / draft_probs[i]
    accept_prob = min(1.0, ratio)
    if ratio >= 1:
        status = "accept (p=1.0)"
    else:
        status = f"accept with p={accept_prob:.0%}"
    print(f"  token {draft_tokens[i]}: p_target/p_draft = {ratio:.2f} -> {status}")

print()
print("The first two tokens have higher target confidence -> always accepted")
print("The third token is not liked by the target -> likely rejected")


### 4. Why does speculative decoding speed things up?

Key fact: the draft model can be 10-100x faster than the target model.

```
Normal decoding:
  1 target forward -> 1 token
  100 tokens -> 100 target forwards

Speculative decoding:
  draft proposes 5 tokens (5 draft forwards, but cheap)
  1 target forward verifies them
  suppose we accept 3 tokens on average
  100 tokens -> about 33 target forwards

Speedup ~= 100/33 ~= 3x
```

The speedup depends on draft accuracy:
- better draft -> more tokens accepted per verification -> faster
- worse draft -> frequent rejection -> smaller speedup (can even get slower)

In practice, a 0.5B draft with a 7B target often yields ~2-3x.


In [ ]:
# Simulate speedup under different acceptance rates
def simulate_speedup(K=5, accept_rate=0.6, num_tokens=100):
    """Approximate the speedup factor of speculative decoding.

    K: how many tokens the draft proposes per step
    accept_rate: per-token acceptance probability
    """
    target_calls = 0
    tokens_generated = 0

    while tokens_generated < num_tokens:
        target_calls += 1
        # A crude expectation: accepted length is capped by K
        expected_accepted = min(K, int(1 / (1 - accept_rate)))
        tokens_generated += expected_accepted

    return num_tokens / target_calls

print("=== Speedup vs acceptance rate (K=5) ===")
print()
for ar in [0.3, 0.5, 0.7, 0.9]:
    speedup = simulate_speedup(K=5, accept_rate=ar)
    print(f"accept rate {ar:.0%}: speedup ~ {speedup:.1f}x")

print()
print("Higher acceptance rate -> more speedup")
print("At 90% acceptance, you approach ~5x (accept ~5 tokens per target call)")


### 5. Variants of speculative decoding

Industry has many variants. The core idea is the same; only "what plays the role of the draft" changes:

| Variant | Draft source | Notes |
|:---|:---|:---|
| **Standard speculative decoding** | a separate small model | requires an extra model to train/deploy |
| **Self-speculative** | early layers of the same model | no extra model; use early-layer predictions |
| **Medusa** | multiple extra heads | attach multiple heads to predict future tokens |
| **Eagle** | a small feature-transform net | predict future tokens from target features |
| **Lookahead decoding** | n-gram matching | use n-grams from generated text as draft |

**Medusa is especially interesting**: attach several small linear heads to the last layer, each predicting the k-th future token.


In [ ]:
# Medusa: a conceptual sketch
print("=== Concept: Medusa heads ===")
print()
print("A standard LLM last layer:")
print("  hidden_states -> lm_head -> predict token_{t+1}")
print()
print("A Medusa-style LLM last layer:")
print("  hidden_states -> lm_head        -> predict token_{t+1}")
print("  hidden_states -> medusa_head_1  -> predict token_{t+2}")
print("  hidden_states -> medusa_head_2  -> predict token_{t+3}")
print("  hidden_states -> medusa_head_3  -> predict token_{t+4}")
print()
print("One forward pass predicts multiple future tokens.")
print("Then the target model verifies them using the same accept/reject idea.")


### 6. Limitations

Not every workload benefits from speculative decoding.

Good fits:
- templated outputs (code, translation, summarization) -> draft is accurate -> high speedup
- server-side large batch inference -> target forwards are expensive

Bad fits:
- creative writing -> draft is inaccurate -> frequent rejections
- single local request -> overhead can outweigh benefits
- tight VRAM budgets -> an extra draft model costs memory

Key insight: speculative decoding does not necessarily reduce total FLOPs; it reduces wall-clock latency by trading "serial waiting" for "parallel verification".


### 7. Normal decoding vs speculative decoding (one diagram)

```
Normal autoregressive decoding:
  Target: [compute]->tok1->[compute]->tok2->[compute]->tok3->[compute]->tok4
  time:    ####          ####          ####          ####
  4 target forwards, fully serial

Speculative decoding:
  Draft:  [compute]->g1->[compute]->g2->[compute]->g3->[compute]->g4   (fast)
  Target:            [verify g1 g2 g3 g4 in one forward]
  time:    ###############################

  If g1,g2 accepted and g3 rejected:
  you get tok1=g1, tok2=g2, tok3=resample
  1 target forward yields 3 tokens -> ~3x
```


---

## Speculative Decoding Summary

1. [ok] Autoregressive serial dependency is a hard constraint.
2. [ok] Speculative decoding: small model proposes tokens, large model verifies in one shot.
3. [ok] Accept/reject uses `min(1, p_target/p_draft)` so the final distribution is unchanged.
4. [ok] Speed depends on draft accuracy; often ~2-3x.
5. [ok] Variants: self-speculative, Medusa, Eagle, lookahead.
6. [ok] Good for templated tasks, not great for creative writing.
7. [ok] It does not eliminate total compute; it reduces wall-clock latency.

One-sentence summary: speculative decoding is like letting a cheap intern draft, and a costly senior only reviews.
